In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/Colab Notebooks/wYe7pBJ7-train.zip" -d "/content/data"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, glob, json, cv2, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torchvision.transforms as T
from PIL import Image

# 1. CẤU HÌNH ĐƯỜNG DẪN GỘP (SỬ DỤNG 100% DATA)
TRAIN_ROOTS = [
    "/content/data/train/Scenario-B/Brazilian",
    "/content/data/train/Scenario-B/Mercosur"
]
WORK_DIR = "/content/drive/MyDrive/LPR_WORK"
os.makedirs(WORK_DIR, exist_ok=True)
CACHE_PATH = os.path.join(WORK_DIR, "corners_multidata_cache.json")
CORNER_CKPT = os.path.join(WORK_DIR, "corner_net.pth")

# 2. HÀM TIỀN XỬ LÝ & MẠNG GÓC
ORDER = ["top-left","top-right","bottom-right","bottom-left"]
get_unique_tid = lambda td: f"{os.path.basename(os.path.dirname(td))}_{os.path.basename(td)}"

def find_tracks(roots):
    if isinstance(roots, str): roots = [roots]
    tracks = []
    for r in roots: tracks.extend(sorted([p for p in glob.glob(os.path.join(r, "track_*")) if os.path.isdir(p)]))
    return tracks

def clean_text(s: str) -> str: return "".join([c for c in (s or "").strip().upper() if c.isalnum()])
def load_ann(td: str):
    with open(os.path.join(td, "annotations.json"), "r", encoding="utf-8") as f: return json.load(f)

def warp_plate(img_bgr, corners: dict, out_h: int):
    src = np.array([corners[k] for k in ORDER], dtype=np.float32)
    tl, tr, br, bl = src
    est_h = max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr))
    est_h = est_h if est_h > 0 else out_h
    out_w = max(8, int(round(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)) * (out_h / est_h))))
    dst = np.array([[0,0],[out_w-1,0],[out_w-1,out_h-1],[0,out_h-1]], dtype=np.float32)
    return cv2.warpPerspective(img_bgr, cv2.getPerspectiveTransform(src, dst), (out_w, out_h), flags=cv2.INTER_CUBIC)

class TinyCornerNet(nn.Module):
    def __init__(self):
        super().__init__()
        block = lambda cin, cout: nn.Sequential(nn.Conv2d(cin, cout, 3, 1, 1), nn.BatchNorm2d(cout), nn.ReLU(True), nn.MaxPool2d(2))
        self.backbone = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 8), nn.Sigmoid())
    def forward(self, x): return self.head(self.backbone(x))

class CornerPredictor:
    def __init__(self, model, device, in_size=(64,32)):
        self.model, self.device, self.wt, self.ht = model.eval(), device, in_size[0], in_size[1]
    @torch.no_grad()
    def predict_corners(self, img_bgr):
        h, w = img_bgr.shape[:2]
        rgb = (cv2.resize(img_bgr, (self.wt, self.ht))[:, :, ::-1].astype(np.float32) / 255.0 - 0.5) / 0.5
        x = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(self.device)
        c = self.model(x).squeeze(0).cpu().numpy().astype(np.float32).reshape(4,2)
        c[:,0] *= float(w); c[:,1] *= float(h)
        return {ORDER[i]: [float(c[i,0]), float(c[i,1])] for i in range(4)}

# 3. TẠO CACHE GÓC (Bỏ qua nhanh nếu đã có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
corner_model = TinyCornerNet().to(device)
if os.path.exists(CORNER_CKPT): corner_model.load_state_dict(torch.load(CORNER_CKPT, map_location=device))

pred = CornerPredictor(corner_model, device)
tracks = find_tracks(TRAIN_ROOTS)
cache = {}
if os.path.exists(CACHE_PATH):
    try:
        with open(CACHE_PATH, "r", encoding="utf-8") as f: cache = json.load(f)
    except: pass

for td in tqdm(tracks, desc="Caching Corners"):
    tid = get_unique_tid(td)
    if tid in cache and len(cache[tid]) > 0: continue
    cache[tid] = {}
    for i in range(1,6):
        names = (f"lr-{i:03d}.jpg", f"lr-{i:03d}.png")
        paths = (os.path.join(td, names[0]), os.path.join(td, names[1]))
        path = paths[0] if os.path.exists(paths[0]) else (paths[1] if os.path.exists(paths[1]) else None)
        img = cv2.imread(path, cv2.IMREAD_COLOR) if path else None
        cache[tid][names[0] if os.path.exists(paths[0]) else names[1]] = pred.predict_corners(img) if img is not None else None

with open(CACHE_PATH, "w", encoding="utf-8") as f: json.dump(cache, f, ensure_ascii=False)
print("✅ Cache nắn góc đã sẵn sàng!")

Caching Corners: 100%|██████████| 10000/10000 [00:00<00:00, 540586.69it/s]


✅ Cache nắn góc đã sẵn sàng!


In [ ]:
class SpatialTransformerNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.loc = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(nn.Linear(64*16, 128), nn.ReLU(True), nn.Linear(128, 6))
        self.fc[2].weight.data.zero_(); self.fc[2].bias.data.copy_(torch.tensor([1,0,0,0,1,0], dtype=torch.float))
    def forward(self, x):
        theta = self.fc(self.loc(x).view(-1, 64*16)).view(-1, 2, 3)
        return F.grid_sample(x, F.affine_grid(theta, x.size(), align_corners=False), align_corners=False)

class ResNetOCR(nn.Module):
    def __init__(self, feat=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, feat, 3, 1, 1), nn.BatchNorm2d(feat), nn.ReLU(True), nn.MaxPool2d((4, 1), (4, 1)),
        )
    def forward(self, x): return self.net(x)

class FrameAttentionFusion(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.att = nn.Sequential(nn.Conv2d(c, c//2, 1), nn.BatchNorm2d(c//2), nn.ReLU(True), nn.Conv2d(c//2, 1, 1))
    def forward(self, x):
        B, N, C, H, W = x.shape
        scores = F.softmax(self.att(x.view(B*N, C, H, W)).view(B, N, 1, H, W), dim=1)
        return torch.sum(x * scores, dim=1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class ResTranOCR_Robust(nn.Module):
    def __init__(self, vocab_size, feat=256, d_model=256, nhead=8, num_layers=3, dropout=0.3):
        super().__init__()
        self.stn, self.cnn, self.fusion = SpatialTransformerNetwork(), ResNetOCR(feat), FrameAttentionFusion(feat)
        self.proj, self.pos_encoder = nn.Linear(feat, d_model), PositionalEncoding(d_model)
        self.transformer = nn.TransformerEncoder(nn.TransformerEncoderLayer(d_model, nhead, d_model*4, batch_first=True, dropout=dropout), num_layers)
        self.dropout, self.fc = nn.Dropout(dropout), nn.Linear(d_model, vocab_size)
    def forward(self, x):
        B, N, C, H, W = x.shape
        f = self.cnn(self.stn(x.view(B*N, C, H, W)))
        fused = self.fusion(f.view(B, N, f.size(1), f.size(2), f.size(3)))
        return self.fc(self.dropout(self.transformer(self.pos_encoder(self.proj(fused.squeeze(2).permute(0, 2, 1))))))

In [ ]:
# 1. SETUP TỪ ĐIỂN VÀ LUẬT
chars = ["<blank>"] + [str(i) for i in range(10)] + [chr(ord("A")+i) for i in range(26)]
stoi, itos = {c: i for i, c in enumerate(chars)}, {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)

def apply_layout_pattern(text, layout):
    t = "".join([c for c in text if c.isalnum()]).upper()
    p = "LLLDLDD" if layout and layout.lower().startswith("mer") else "LLLDDDD"
    if len(t) < 7: t += "0"*(7-len(t))
    t = t[:7]
    to_l = lambda c: {"0":"O","1":"I","2":"Z","5":"S","8":"B","4":"A","7":"T","6":"G"}.get(c,c) if c.isdigit() else c
    to_d = lambda c: {"O":"0","I":"1","Z":"2","S":"5","B":"8","D":"0","A":"4","G":"6","T":"7","Q":"0"}.get(c,c) if c.isalpha() else c
    return "".join([to_l(c) if px == "L" else to_d(c) for c, px in zip(t, p)])

# 2. DATASET CO CHỮA AUGMENTATION
train_transform = T.Compose([T.ColorJitter(0.4,0.4,0.4), T.GaussianBlur((3,3), (0.1,1.5)), T.RandomRotation(5)])
def to_tensor_aug(img, is_train):
    rgb = img[:,:,::-1]
    if is_train: rgb = np.array(train_transform(Image.fromarray(rgb)))
    return torch.from_numpy((rgb.astype(np.float32)/255.0 - 0.5)/0.5).permute(2,0,1)

class MultiFrameOCRDatasetCached(Dataset):
    def __init__(self, tracks, cache_path, th=32, tw=128, is_train=False):
        self.tracks, self.th, self.tw, self.is_train = tracks, th, tw, is_train
        with open(cache_path, "r", encoding="utf-8") as f: self.cache = json.load(f)
    def __len__(self): return len(self.tracks)
    def __getitem__(self, idx):
        td, tid = self.tracks[idx], get_unique_tid(self.tracks[idx])
        ann = load_ann(td)
        lay, txt = ann.get("plate_layout", "Mercosur"), clean_text(ann.get("plate_text", ""))
        frames = []
        for i in range(1,6):
            p1, p2 = os.path.join(td, f"lr-{i:03d}.jpg"), os.path.join(td, f"lr-{i:03d}.png")
            img = cv2.imread(p1) if os.path.exists(p1) else (cv2.imread(p2) if os.path.exists(p2) else None)
            c = self.cache.get(tid, {}).get(f"lr-{i:03d}.jpg") or self.cache.get(tid, {}).get(f"lr-{i:03d}.png")
            if img is None or c is None: frames.append(torch.zeros(3, self.th, self.tw))
            else: frames.append(to_tensor_aug(cv2.resize(warp_plate(img, c, self.th), (self.tw, self.th)), self.is_train))
        return torch.stack(frames, dim=0), txt, lay, tid

# 3. TRAINING LOOP
def collate(b): return torch.stack([x[0] for x in b]), [x[1] for x in b], [x[2] for x in b], [x[3] for x in b]
def ctc_decode(logits):
    idx = torch.softmax(logits, dim=-1).argmax(dim=-1)
    res = []
    for b in range(idx.size(0)):
        out, prev = [], None
        for t in range(idx.size(1)):
            c = int(idx[b,t])
            if c != 0 and c != prev: out.append(itos[c])
            prev = c
        res.append("".join(out))
    return res

random.seed(42)
all_tracks = find_tracks(TRAIN_ROOTS)
random.shuffle(all_tracks)
tr_tracks, va_tracks = all_tracks[:int(0.95*len(all_tracks))], all_tracks[int(0.95*len(all_tracks)):]
print(f"📦 Khởi động Training: {len(tr_tracks)} Train | {len(va_tracks)} Val")

tr_loader = DataLoader(MultiFrameOCRDatasetCached(tr_tracks, CACHE_PATH, is_train=True), 32, True, num_workers=2, collate_fn=collate)
va_loader = DataLoader(MultiFrameOCRDatasetCached(va_tracks, CACHE_PATH, is_train=False), 32, False, num_workers=2, collate_fn=collate)

model = ResTranOCR_Robust(vocab_size).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=5e-4, steps_per_epoch=len(tr_loader), epochs=40, pct_start=0.2)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)
best_acc, best_path = 0.0, os.path.join(WORK_DIR, "best_multidomain_explicit.pth")

for ep in range(1, 100):
    model.train(); tl = 0.0
    for X, txts, lays, _ in tqdm(tr_loader, desc=f"Epoch {ep}/100"):
        logits = model(X.to(device))
        targs, tlens = [], []
        for t in txts:
            seq = [stoi[c] for c in t if c in stoi]; targs.extend(seq); tlens.append(len(seq))
        logp = logits.log_softmax(-1).permute(1,0,2)
        loss = ctc(logp, torch.tensor(targs).to(device), torch.full((logp.size(1),), logp.size(0)), torch.tensor(tlens).to(device))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); scheduler.step(); tl+=loss.item()

    model.eval(); correct, total = 0, 0
    with torch.no_grad():
        for X, txts, lays, _ in va_loader:
            preds = ctc_decode(model(X.to(device)))
            for p, gt, lay in zip(preds, txts, lays):
                if apply_layout_pattern(p, lay) == clean_text(gt): correct += 1
                total += 1
    acc = correct / max(1, total)
    print(f"✅ Ep {ep}: Loss={tl/len(tr_loader):.4f} | VAL ACC={acc*100:.2f}%")
    if acc > best_acc: best_acc = acc; torch.save(model.state_dict(), best_path); print(" 💾 Đã lưu model xịn nhất!")

📦 Khởi động Training: 9500 Train | 500 Val


Epoch 1/40: 100%|██████████| 297/297 [01:31<00:00,  3.23it/s]


✅ Ep 1: Loss=3.8490 | VAL ACC=0.00%


Epoch 2/40: 100%|██████████| 297/297 [01:30<00:00,  3.27it/s]


✅ Ep 2: Loss=3.0527 | VAL ACC=0.00%


...

✅ Ep 38: Loss=0.0511 | VAL ACC=69.60%


Epoch 39/40: 100%|██████████| 297/297 [01:32<00:00,  3.21it/s]


✅ Ep 39: Loss=0.0500 | VAL ACC=69.80%


Epoch 40/40: 100%|██████████| 297/297 [01:32<00:00,  3.21it/s]


✅ Ep 40: Loss=0.0493 | VAL ACC=69.80%


In [ ]:
BLIND_TEST_ROOT = "/content/data/TKzFBtn7-test-blind"
OUT_FILE = os.path.join(WORK_DIR, "submission_blind_test.txt")

def generate_blind_submission():
    print("🚀 Đang khởi động Inference pipeline...")
    c_model = TinyCornerNet().to(device)
    c_model.load_state_dict(torch.load(CORNER_CKPT, map_location=device))
    c_pred = CornerPredictor(c_model, device)

    o_model = ResTranOCR_Robust(vocab_size).to(device)
    if os.path.exists(best_path): o_model.load_state_dict(torch.load(best_path, map_location=device))
    o_model.eval()

    test_tracks = find_tracks(BLIND_TEST_ROOT)
    results = []

    for td in tqdm(test_tracks, desc="Dự đoán Blind Test"):
        tid = os.path.basename(td)
        tid = tid[tid.find("track_"):] if "track_" in tid else tid

        lay = None
        if os.path.exists(os.path.join(td, "annotations.json")):
            try: lay = json.load(open(os.path.join(td, "annotations.json")))["plate_layout"]
            except: pass

        frames = []
        for i in range(1, 6):
            p1, p2 = os.path.join(td, f"lr-{i:03d}.jpg"), os.path.join(td, f"lr-{i:03d}.png")
            img = cv2.imread(p1) if os.path.exists(p1) else (cv2.imread(p2) if os.path.exists(p2) else None)
            if img is None: frames.append(torch.zeros(3, 32, 128)); continue

            c = c_pred.predict_corners(img)
            rgb = (cv2.resize(warp_plate(img, c, 32), (128, 32))[:,:,::-1].astype(np.float32)/255.0 - 0.5)/0.5
            frames.append(torch.from_numpy(rgb).permute(2,0,1))

        X = torch.stack(frames, dim=0).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = o_model(X)
            probs = torch.softmax(logits, dim=-1)
            idx, conf = probs.argmax(-1)[0], probs.max(-1).values[0]
            out, out_c, prev = [], [], None
            for t in range(idx.size(0)):
                c = int(idx[t])
                if c != 0 and c != prev: out.append(itos[c]); out_c.append(float(conf[t]))
                prev = c

            raw_p, avg_conf = "".join(out), sum(out_c)/max(1, len(out_c)) if out_c else 0.0
            final_p = apply_layout_pattern(raw_p, lay) if lay else clean_text(raw_p) if raw_p else "BLANK"
            results.append(f"{tid},{final_p};{avg_conf:.4f}\n")

    with open(OUT_FILE, "w", encoding="utf-8") as f: f.writelines(results)
    print(f"✅ Xong! File nộp bài đã có tại: {OUT_FILE}")

generate_blind_submission()